# 08 · From a datacube to an answer

Notebook `04` reduced each pass to one number by averaging **its own** grid.
That makes an honest-looking trend line out of slightly dishonest arithmetic:
successive Umbra passes over a site are delivered in whatever UTM zone and
extent each acquisition happened to use, so pass 1's mean and pass 4's mean
describe *different ground*. Move the footprint a little and the "change" moves
with it.

This is also exactly where the standard STAC stack gives up. `stackstac` and
`odc-stac` want a common projected grid and assume a STAC *API*; Umbra's open
data is a static catalog of scenes that were never put on a shared grid. So
`umbra_py` does the alignment itself:

- **`to_stack`** warps every pass onto *one* grid and returns a labelled
  `(time, y, x)` `xarray.DataArray` — a datacube where `cube[3, 40, 12]` and
  `cube[0, 40, 12]` are the same square metres of ground on two dates.
- **`stack_stats`** reduces that cube to a small JSON document: what each pass
  measured, how much the site moved between passes, **where** it moved, and
  **when** — the answer, not the array.

We'll go search → cube → answer, and finish by mapping the baseline-to-latest
decibel delta.

This notebook needs the `load` extra (xarray + rasterio + numpy):

```bash
pip install "umbra-py[load]"
```

> *Contains Umbra open data, licensed under CC BY 4.0.*

## 1 · Find a site imaged several times

An Umbra *task* is repeat imaging of one site, so grouping a modest search by
task and taking the busiest task gives us a series to stack — the same opening
move as notebooks `03` and `04`.

In [ ]:
from umbra_py import UmbraCatalog

by_task: dict[str, list] = {}
for it in UmbraCatalog().search(
    start="2024-01-01", end="2024-12-31", product_types=["GEC"], limit=24
):
    if "GEC" in it.available_assets and it.task and it.datetime is not None:
        by_task.setdefault(it.task, []).append(it)

series = max(by_task.values(), key=len) if by_task else []
assert len(series) >= 2, "no repeat-imaged GEC task in the sampled window"
print(f"{series[0].task}: {len(series)} dated GEC passes")

## 2 · One polarization, oldest first

A cube is a time axis, and anything that varies along it reads as change — so an
HH pass sitting between two VV passes would show up as a site that brightened
and then dimmed. `select_change_frames(..., frames=None)` returns the whole
series oldest-first *after* collapsing it to a single polarization, which is the
guard `to_stack`'s docstring asks callers to apply.

We also cap the series here. Every pass is a warp plus a decimated read, and the
cube is held in memory, so cost grows with the number of passes; six is plenty
to tell a drift from a step.

In [ ]:
from umbra_py import select_change_frames

# frames=None keeps every pass (oldest-first), grouped to one polarization.
passes = select_change_frames(series, frames=None)[:6]
assert len(passes) >= 2

pol = "/".join(passes[0].polarizations) or "?"
print(f"stacking {len(passes)} passes, polarization {pol}")
for p in passes:
    print("  ", p.datetime.date(), p.id)

## 3 · Co-register the passes onto one equal-area grid

`to_stack` opens each GEC, warps it to a shared CRS, and reads a decimated
overview through HTTP range requests — no full download, and no multi-gigabyte
temp files.

Two arguments carry the weight:

- **`crs=STACK_AUTO_CRS`** (`"utm"`) picks the UTM zone containing the stacked
  ground, so cells are square and metre-sized. The lon/lat default is fine for
  looking, but its cells are not equal-area, and counting them would not be
  measuring. Since we want an answer in km², we project.
- **`extent="intersection"`** keeps only the ground *every* pass covered, so no
  cell has a gap. It raises if the footprints don't all overlap — passes of one
  task usually do, but when they don't, `"union"` keeps all the ground any pass
  covered and leaves the rest `NaN`.

`db=True` returns the decibel scale, where a ratio of backscatter becomes a
subtraction — the scale differencing should happen on.

In [ ]:
from umbra_py import STACK_AUTO_CRS, to_stack

kwargs = dict(asset="GEC", max_size=384, db=True, crs=STACK_AUTO_CRS)
try:
    cube = to_stack(passes, extent="intersection", **kwargs)
except ValueError as exc:  # footprints don't all overlap
    print(f"intersection unavailable ({exc}); falling back to union")
    cube = to_stack(passes, extent="union", **kwargs)

assert cube.dims == ("time", "y", "x")
assert cube.sizes["time"] == len(passes)
xres, _, _, _, yres, _ = cube.attrs["transform"]
print(f"cube {dict(cube.sizes)} on {cube.attrs['crs']} in {cube.attrs['units']}")
print(f"cell {abs(xres):.1f} x {abs(yres):.1f} (projected units, so equal-area)")
print("span:", str(cube["time"].values[0])[:10], "->", str(cube["time"].values[-1])[:10])

It is an ordinary `xarray.DataArray` from here on — `cube.mean("time")`,
`cube.std("time")` and `cube.diff("time")` are now per-ground-cell statistics
rather than per-scene ones, and the `item_id` coordinate keeps every slice's
provenance attached.

In [ ]:
# Per-cell statistics across the series, which is what co-registration bought.
print("mean over time :", round(float(cube.mean("time").mean()), 2), "dB")
print("std  over time :", round(float(cube.std("time").mean()), 2), "dB")
print("item ids       :", [str(v) for v in cube["item_id"].values][:3], "...")

## 4 · Reduce the cube to an answer

`stack_stats` walks the time axis and returns plain JSON: one record per pass
(its distribution, plus the signed change against the pass before it) and one
net baseline → latest record. Change is *always* reported in decibels, whatever
scale the cube holds, so the numbers mean the same thing either way.

Because the grid is projected, `changed_area_km2` is a real area — on a lon/lat
cube it would be `None`, and the document says so in its own `caveats`.

In [ ]:
from umbra_py import stack_stats

stats = stack_stats(cube)
assert stats["count"] == len(passes)

for rec in stats["passes"]:
    step = rec["change_vs_previous"]
    trend = f"{step['mean_delta_db']:+6.2f} dB vs. previous" if step else "baseline"
    print(f"{rec['datetime'][:10]}  mean {rec['mean']:7.2f} dB   {trend}")

net = stats["net_change"]
print(
    f"\nnet first->last: {net['mean_delta_db']:+.2f} dB, "
    f"{net['changed_fraction']:.1%} of cells past "
    f"{stats['change_threshold_db']} dB ({net['changed_area_km2']} km2)"
)

## 5 · Say *where*, not just how much

A scene-wide mean dilutes a localized change: one corner brightening 12 dB reads
as 0.75 dB averaged over sixteen blocks. `blocks=N` cuts every pass into the same
N×N grid — the grid `umbra change --narrate` uses for two passes — so each block
reports its own net change, a compass label, a `center_lonlat` to map or geocode
it by, and the consecutive interval it moved most in.

`grid_text` renders the whole grid north-up as ASCII, which is a surprisingly
good way to *see* the answer in a terminal (and the reason an agent can read it
without an image).

In [ ]:
stats = stack_stats(cube, blocks=3, block_series=True)
spatial = stats["spatial"]

peak = spatial["peak_block"]
print(f"peak block: {peak['compass']} at {peak['center_lonlat']}")
print(f"  net {peak['mean_delta_db']:+.2f} dB over the series ({peak['direction']})")
when = peak["peak_interval"]
print(
    f"  moved most {when['from_datetime'][:10]} -> {when['to_datetime'][:10]}"
    f" ({when['mean_delta_db']:+.2f} dB)"
)

print("\nnet signed change per block (north-up):")
print(spatial["grid_text"])

## 6 · Steady drift, or one step and hold?

`peak_interval` is one number, and one number cannot separate two genuinely
different histories: a corner that drifts a decibel every pass and a corner that
jumped twelve once and held both report the same net change and the same peak.
`block_series=True` (asked for above) keeps the sequence those peaks were picked
from — every consecutive step, oldest first — so the *shape* of a block's
history is readable rather than inferred.

In [ ]:
block = next(b for b in spatial["blocks"] if (b["row"], b["col"]) == (peak["row"], peak["col"]))
steps = block["series"]
# One record per consecutive pair -- fewer only if a pair shared no observed cell.
assert len(steps) <= len(passes) - 1

print(f"{block['compass']} block, pass to pass:")
for s in steps:
    bar = "#" * min(int(abs(s["mean_delta_db"]) * 2), 40)
    print(
        f"  {s['from_datetime'][:10]} -> {s['to_datetime'][:10]}"
        f"  {s['mean_delta_db']:+6.2f} dB  {bar}"
    )

# One big step among small ones is an event; a run of similar steps is a drift.
largest = max(steps, key=lambda s: abs(s["mean_delta_db"]), default=None)
if largest is not None:
    assert largest["mean_delta_db"] == when["mean_delta_db"], "peak is a member of the series"
    print(f"\nlargest single step: {largest['mean_delta_db']:+.2f} dB")

## 7 · Map the delta

The numbers came out of the cube, so the picture can too: subtract the baseline
slice from the latest one and you have a per-cell decibel difference on a
metre-sized grid, ready to plot. Red is brighter than baseline, blue is dimmer —
the same reading the block grid above gave in text.

In [ ]:
delta = cube.isel(time=-1) - cube.isel(time=0)  # dB difference, per ground cell

try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(6, 5))
    img = ax.imshow(delta.values, cmap="RdBu_r", vmin=-6, vmax=6)
    ax.set_title(f"{series[0].task} — latest minus baseline")
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(img, ax=ax, label="change (dB)")
    plt.show()
except ImportError:
    print("(install matplotlib to map the delta)")
    print(f"delta range: {float(delta.min()):.1f} .. {float(delta.max()):.1f} dB")

## Where next

Everything above has a one-line command-line equivalent, and the same reduction
is reachable from an agent:

```bash
# the cube as a multi-band GeoTIFF, plus the answer as JSON
umbra stack --area "<site>" --crs utm --out cube.tif --stats --blocks 3 --block-series

# no file at all — just measure
umbra stack --area "<site>" --stats --blocks 3
```

- **`stack_to_geotiff`** writes the same cube as a multi-band GeoTIFF for QGIS
  or anything else that reads rasters.
- **`stack_stats`** is a tool on the MCP server and on the LangChain and
  LlamaIndex wrappers, and `POST /artifacts/stats` on `umbra serve` answers it
  over HTTP — the `umbra demo` explorer's **Quantify** button is that endpoint
  with the numbers drawn as sparklines.
- **`umbra change --narrate`** (the `ai` extra) hands a composite and its dB grid
  to a vision model, which may only describe change the numbers support.
- **Notebook `04`** is the scalar version of this one, and **`06`** turns a site
  into a standing monitor that runs on a schedule.

A closing caveat the document carries too: Umbra's open products are not
radiometrically calibrated, so decibel values are *relative*. Compare a cell to
itself across dates — which is what this whole notebook does — not to another
site or sensor.

*Contains Umbra open data, licensed under CC BY 4.0.*